In [5]:
%pip install scikit-learn

Note: you may need to restart the kernel to use updated packages.


In [6]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.metrics import (
    accuracy_score, confusion_matrix, ConfusionMatrixDisplay,
    classification_report, roc_auc_score, roc_curve,
    mean_squared_error, mean_absolute_error, r2_score
)
from sklearn.preprocessing import StandardScaler

sns.set_theme(style="whitegrid")
pd.set_option('display.max_columns', None)
df = pd.read_csv("Titanic-Dataset.csv")
df.shape

(891, 12)

In [7]:
df_clean = df.drop(columns=["PassengerId", "Name", "Ticket", "Cabin"])

df_clean["Age"] = df_clean["Age"].fillna(df_clean["Age"].median())
df_clean["Embarked"] = df_clean["Embarked"].fillna(df_clean["Embarked"].mode()[0])

df_clean["FamilySize"] = df_clean["SibSp"] + df_clean["Parch"] + 1
df_clean["IsAlone"] = (df_clean["FamilySize"] == 1).astype(int)
df_clean["LogFare"] = np.log1p(df_clean["Fare"])

df_clean["Sex"] = df_clean["Sex"].map({"male": 0, "female": 1})
df_clean["Embarked"] = df_clean["Embarked"].map({"S": 0, "C": 1, "Q": 2})

df_clean = df_clean.drop(columns=["SibSp", "Parch", "Fare"])

print("Missing values remaining:", df_clean.isnull().sum().sum())
print("Shape after prep:", df_clean.shape)
df_clean.head()

Missing values remaining: 0
Shape after prep: (891, 8)


,Survived,Pclass,Sex,Age,Embarked,FamilySize,IsAlone,LogFare
0,0,3,0,22.0,0,2,0,2.110213
1,1,1,1,38.0,1,2,0,4.280593
2,1,3,1,26.0,0,1,1,2.188856
3,1,1,1,35.0,0,2,0,3.990834
4,0,3,0,35.0,0,1,1,2.202765


In [19]:
FEATURES = ["Pclass", "Sex", "Age", "LogFare", "Embarked", "FamilySize", "IsAlone"]
TARGET = "Survived"

X = df_clean[FEATURES]
y = df_clean[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

print(f"Training set:  {X_train.shape[0]} rows")
print(f"Test set:      {X_test.shape[0]} rows")
print(f"\nSurvival rate in train: {y_train.mean():.3f}")
print(f"Survival rate in test:  {y_test.mean():.3f}")

Training set:  668 rows
Test set:      223 rows

Survival rate in train: 0.383
Survival rate in test:  0.386


In [20]:
majority_class = y_train.mode()[0]
naive_preds = np.full(len(y_test), majority_class)
naive_accuracy = accuracy_score(y_test, naive_preds)

print(f"Majority class: {majority_class} (Did not survive)")
print(f"Naive baseline accuracy: {naive_accuracy:.3f}")
print()
print("Interpretation: A model that always predicts 'did not survive'")
print(f"would be correct {naive_accuracy*100:.1f}% of the time — without learning anything.")
print(f"Our model must beat {naive_accuracy:.3f} to be worth using.")

Majority class: 0 (Did not survive)
Naive baseline accuracy: 0.614

Interpretation: A model that always predicts 'did not survive'
would be correct 61.4% of the time — without learning anything.
Our model must beat 0.614 to be worth using.
